# Evaluate Trade Demo: The Final Phase 6 Integration Point

`src/trade_engine/evaluate_trade.py` is where every piece built tonight actually meets: given two real rosters and a proposed trade, it

1. calls `net_value.py`'s `evaluate_player_trade_value` for every player moving on both sides (predicted KVS, keeper cost, raw `net_kvs_delta`, plus that player's own confidence caveats),
2. adjusts each player's raw delta with `positional_need.py`'s `apply_positional_need_adjustment`, using the **receiving** team's own current roster,
3. sums each side into a total adjusted value,
4. runs `fairness_score.py`'s `compute_fairness_score` for the headline percentage-of-total verdict, and
5. calls `src/explain/explain_player.py`'s `explain_player` for the 1-2 players on each side actually driving that side's total, so the verdict comes with real SHAP-based "why", not just a number.

Every caveat any player carries (`low_confidence_extreme_delta`, `no_delta_history`, `empty_position`) is rolled up into the final result, tagged with exactly which side and player it belongs to -- never dropped at this last integration layer.

**A real gap found and fixed to make this possible**: Phase 5's own roadmap exit criterion called for `explain_player(player, season) -> top_5_drivers` as a *reusable function*. What existed was four byte-identical copies of that function pasted into `notebooks/08a-d_shap_*.ipynb`, one per position, never promoted into an importable module -- `src/explain/` sat in the repo completely empty. `src/explain/explain_player.py` is that promotion, built when `evaluate_trade.py` needed a real, callable `explain_player` and the gap went from cosmetic to blocking.

Two real trades between actual current rostered players (`data/processed/roster_keeper_table_2027.csv`) run end to end below -- real models, real SHAP, real roster construction, nothing mocked.

## Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.trade_engine.draft_capital_curve import build_full_draft_capital_curve, normalize_player_name
from src.trade_engine.evaluate_trade import TradeAsset, evaluate_trade
from src.trade_engine.fairness_score import describe_fairness

pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

SEASON = 2025  # each player's real, already-realized 2025 season feeds the 2026 prediction,
               # matching every 06x_model_*.ipynb / 08x_shap_*.ipynb notebook's own convention.

draft_curve = build_full_draft_capital_curve(REPO_ROOT)
vorp_labels = pd.read_parquet(REPO_ROOT / "data/processed/vorp_labels.parquet")
roster_table = pd.read_csv(REPO_ROOT / "data/processed/roster_keeper_table_2027.csv")
print(f"draft_curve: {len(draft_curve)} rounds, roster_table: {len(roster_table)} rostered players across {roster_table['owner'].nunique()} real teams")

draft_curve: 18 rounds, roster_table: 183 rostered players across 10 real teams


## Building real rosters, with `scarcity_z`, for positional need adjustment

`positional_need.py` expects a roster as a list of `{"position": ..., "scarcity_z": ...}` dicts -- the receiving team's OWN players, so `apply_positional_need_adjustment` can see how deep or thin that team already is at the position it's acquiring. `roster_keeper_table_2027.csv` only has player names, so each roster is built by joining against `vorp_labels`' 2025 season, using `draft_capital_curve.py`'s own `normalize_player_name` (the same suffix/punctuation-insensitive matching already validated for the draft-pick crosswalk) plus a team-name-to-code lookup for DEF rows, since `vorp_labels` keys defenses by team code, not full team name.

Not every rostered player resolves -- 2025 rookies who logged too little to get a `vorp_labels` row (or genuine name-format misses) are skipped rather than guessed at, the same "an unresolved miss is a known, visible gap; a silently wrong match is worse" principle `draft_capital_curve.py`'s crosswalk already holds itself to.

In [2]:
import nflreadpy as nfl

v2025 = vorp_labels[vorp_labels["season"] == SEASON][["position", "player_display_name", "vorp"]].copy()
v2025["norm_name"] = v2025["player_display_name"].apply(normalize_player_name)
position_stats = v2025.groupby("position")["vorp"].agg(mean="mean", std="std")
v2025 = v2025.merge(position_stats, on="position", how="left")
v2025["scarcity_z"] = (v2025["vorp"] - v2025["mean"]) / v2025["std"]

teams = nfl.load_teams().to_pandas()
team_name_to_code = dict(zip(teams["team_name"], teams["team_abbr"]))
def_rows_2025 = vorp_labels[(vorp_labels["season"] == SEASON) & (vorp_labels["position"] == "DEF")][["player_id", "vorp"]]
def_mean = v2025.loc[v2025["position"] == "DEF", "mean"].iloc[0]
def_std = v2025.loc[v2025["position"] == "DEF", "std"].iloc[0]


def build_roster_with_scarcity_z(owner: str) -> list:
    # A real team's current roster, as positional_need.py expects it:
    # [{"position": ..., "scarcity_z": ...}, ...]
    roster = []
    unmatched = []
    for _, row in roster_table[roster_table["owner"] == owner].iterrows():
        position, player = row["position"], row["player"]
        if position == "DEF":
            code_ = team_name_to_code.get(player)
            match = def_rows_2025[def_rows_2025["player_id"] == code_] if code_ else pd.DataFrame()
            if match.empty:
                unmatched.append(player)
                continue
            z = (match["vorp"].iloc[0] - def_mean) / def_std
            roster.append({"position": "DEF", "scarcity_z": float(z)})
            continue
        norm_name = normalize_player_name(player)
        match = v2025[(v2025["position"] == position) & (v2025["norm_name"] == norm_name)]
        if match.empty:
            unmatched.append(player)
            continue
        roster.append({"position": position, "scarcity_z": float(match["scarcity_z"].iloc[0])})
    if unmatched:
        print(f"  {owner}: {len(unmatched)} unmatched (likely 2025 rookies with no vorp_labels row yet): {unmatched}")
    return roster


rosters = {owner: build_roster_with_scarcity_z(owner) for owner in ["Aumr11", "ChocoLitMilk", "HarryUncle"]}
for owner, roster in rosters.items():
    print(f"{owner}: {len(roster)} players matched")

  Aumr11: 2 unmatched (likely 2025 rookies with no vorp_labels row yet): ['Denzel Boston', 'Makai Lemon']
  ChocoLitMilk: 3 unmatched (likely 2025 rookies with no vorp_labels row yet): ["Ja'Kobi Lane", 'Jeremiyah Love', 'Trey Smack']
  HarryUncle: 1 unmatched (likely 2025 rookies with no vorp_labels row yet): ['Jonah Coleman']
Aumr11: 16 players matched
ChocoLitMilk: 15 players matched
HarryUncle: 17 players matched


### Trade report helper

One formatting function, reused for both trades below, so the two real runs are directly comparable.

In [3]:
def print_trade_report(result, team_a_name, team_b_name):
    print("=" * 78)
    print(f"FAIRNESS VERDICT  (method: {result.fairness.fairness_method}{', shifted' if result.fairness.shifted else ''})")
    print(f"  {describe_fairness(result.fairness, team_a_name=team_a_name)}")
    print(f"  {team_a_name:20s} total adjusted value: {result.team_a_total:+8.2f}")
    print(f"  {team_b_name:20s} total adjusted value: {result.team_b_total:+8.2f}")
    print()

    for side_name, players in [(team_a_name, result.team_a_players), (team_b_name, result.team_b_players)]:
        print(f"-- {side_name} receives --")
        for p in players:
            print(f"  {p.player_name:22s} ({p.position})  predicted_KVS={p.predicted_kvs:8.2f}  "
                  f"keeper_cost={p.keeper_cost_vorp:8.2f}  raw_delta={p.raw_net_kvs_delta:8.2f}  "
                  f"need_z={p.team_need_z:+5.2f}  adjustment={p.adjustment_vorp:+8.2f}  "
                  f"ADJUSTED={p.adjusted_net_kvs_delta:+8.2f}")
            for c in p.caveats:
                print(f"      CAVEAT: {c}")
        print()

    for side_name, drivers in [(team_a_name, result.team_a_explained_drivers), (team_b_name, result.team_b_explained_drivers)]:
        print(f"-- Why {side_name}'s biggest mover(s) look the way they do (real SHAP) --")
        for d in drivers:
            print(f"  {d.player_name} ({d.position}), adjusted_net_kvs_delta={d.adjusted_net_kvs_delta:+.2f}")
            if d.explanation_error:
                print(f"    explanation FAILED (captured, did not abort the trade eval): {d.explanation_error}")
            else:
                for fd in d.top_drivers:
                    print(f"    {fd.feature:24s} value={fd.feature_value:8.3f}   shap={fd.shap_value:+7.3f}   ({fd.direction})")
        print()

    if result.caveats:
        print(f"-- All caveats, attributed (never dropped) --")
        for c in result.caveats:
            print(f"  [{c.side}] {c.player_name}: {c.caveat}")
    else:
        print("-- No caveats on either side --")

## Trade 1: Aumr11 <-> ChocoLitMilk (1-for-1)

Aumr11 sends **Terry McLaurin (WR)** to ChocoLitMilk for **J.K. Dobbins (RB)**. Both owners are already deep at the position they'd be receiving (Aumr11 rosters 5 RBs, ChocoLitMilk rosters 6 WRs) -- exactly the kind of real, plausible-but-not-obviously-smart trade fantasy managers actually propose, and a genuine test of whether positional need correctly discounts an addition to an already-strong position rather than only ever rewarding need.

**Simplification, stated explicitly rather than silently assumed**: `projected_keeper_round` uses each player's CURRENT owner's `round_lost_if_kept_2027` from `roster_keeper_table_2027.csv` as a stand-in for what the *acquiring* team would pay to keep them. Real keeper-cost inheritance after a trade is a separate rules question this demo does not model -- `evaluate_trade` itself takes `projected_keeper_round` as a plain input per `TradeAsset`, agnostic to how a caller sources it.

In [4]:
trade1_team_a_players = [
    TradeAsset("J.K. Dobbins", SEASON, "RB",
               projected_keeper_round=int(roster_table.loc[(roster_table["owner"] == "ChocoLitMilk") & (roster_table["player"] == "J.K. Dobbins"), "round_lost_if_kept_2027"].iloc[0])),
]
trade1_team_b_players = [
    TradeAsset("Terry McLaurin", SEASON, "WR",
               projected_keeper_round=int(roster_table.loc[(roster_table["owner"] == "Aumr11") & (roster_table["player"] == "Terry McLaurin"), "round_lost_if_kept_2027"].iloc[0])),
]

trade1_result = evaluate_trade(
    trade1_team_a_players, trade1_team_b_players,
    rosters["Aumr11"], rosters["ChocoLitMilk"],
    draft_curve, REPO_ROOT, vorp_labels=vorp_labels,
)
print_trade_report(trade1_result, "Aumr11", "ChocoLitMilk")

C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-Test\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FAIRNESS VERDICT  (method: magnitude_relative_both_negative)
  Aumr11 receives 56% of the trade's total value.
  Aumr11               total adjusted value:   -62.59
  ChocoLitMilk         total adjusted value:   -80.78

-- Aumr11 receives --
  J.K. Dobbins           (RB)  predicted_KVS=  -37.71  keeper_cost=  -10.86  raw_delta=  -26.85  need_z=+1.65  adjustment=  -35.74  ADJUSTED=  -62.59

-- ChocoLitMilk receives --
  Terry McLaurin         (WR)  predicted_KVS=  -10.40  keeper_cost=   49.86  raw_delta=  -60.26  need_z=+1.29  adjustment=  -20.52  ADJUSTED=  -80.78
      CAVEAT: low_confidence_extreme_delta: |vorp_delta_yoy| > 100 -- this region has confirmed, held-out degraded accuracy at every modeled position (see 08a-08d_shap_*.ipynb); treat the exact predicted_KVS number with real skepticism, not just the direction.

-- Why Aumr11's biggest mover(s) look the way they do (real SHAP) --
  J.K. Dobbins (RB), adjusted_net_kvs_delta=-62.59
    draft_pick_inverse       value=   0.018   s

**Real-data eyeball check, Trade 1**: both sides come back negative (`team_a_total = -62.59`, `team_b_total = -80.78`) -- the raw model already reads both players as worth less than their keeper cost, and positional need makes it WORSE on both sides (Aumr11's RB need_z = +1.65, ChocoLitMilk's WR need_z = +1.29, both clearly above-average depth, so each discount pushes further negative rather than offsetting it). This is `compute_fairness_score`'s both-negative fallback (`magnitude_relative_both_negative`) firing on real data, not just the unit tests: with two real losses of different sizes, it reports `56.3% / 43.7%` -- ChocoLitMilk is the relatively-less-bad side, not the near-0%/100% the old shift-based math would have produced for values this close in magnitude. Terry McLaurin correctly still carries `low_confidence_extreme_delta`, and the SHAP explanation for both top movers ran cleanly with no `explanation_error`. **Read plainly: the system is telling both managers this isn't a good trade for either of them** -- a real, useful verdict, not just a "trades are complicated" shrug.

## Trade 2: Aumr11 <-> HarryUncle (2-for-2 blockbuster)

Aumr11 sends **Christian McCaffrey (RB)** and **Jaxson Dart (QB)** to HarryUncle for **Justin Jefferson (WR)** and **Rhamondre Stevenson (RB)**. Deliberately includes McCaffrey -- already documented (`06b_model_rb.ipynb`/`08b_shap_rb.ipynb`) as carrying a real `low_confidence_extreme_delta` flag -- and Dart, a true rookie with no prior season (`no_delta_history`), to confirm both caveat types survive all the way through `evaluate_trade`'s full pipeline, not just `net_value.py` in isolation. Two players per side also exercises real top-mover selection (`top_n_explained`, default 2) rather than the trivial one-player case.

In [5]:
def keeper_round(owner, player_name):
    match = roster_table[(roster_table["owner"] == owner) & (roster_table["player"] == player_name)]
    return int(match.iloc[0]["round_lost_if_kept_2027"])


trade2_team_a_players = [
    TradeAsset("Justin Jefferson", SEASON, "WR", keeper_round("HarryUncle", "Justin Jefferson")),
    TradeAsset("Rhamondre Stevenson", SEASON, "RB", keeper_round("HarryUncle", "Rhamondre Stevenson")),
]
trade2_team_b_players = [
    TradeAsset("Christian McCaffrey", SEASON, "RB", keeper_round("Aumr11", "Christian McCaffrey")),
    TradeAsset("Jaxson Dart", SEASON, "QB", keeper_round("Aumr11", "Jaxson Dart")),
]

trade2_result = evaluate_trade(
    trade2_team_a_players, trade2_team_b_players,
    rosters["Aumr11"], rosters["HarryUncle"],
    draft_curve, REPO_ROOT, vorp_labels=vorp_labels,
)
print_trade_report(trade2_result, "Aumr11", "HarryUncle")

FAIRNESS VERDICT  (method: magnitude_relative_both_negative)
  Aumr11 receives 20% of the trade's total value.
  Aumr11               total adjusted value:  -211.95
  HarryUncle           total adjusted value:   -54.24

-- Aumr11 receives --
  Justin Jefferson       (WR)  predicted_KVS=   17.98  keeper_cost=  107.36  raw_delta=  -89.38  need_z=+0.95  adjustment=  -15.08  ADJUSTED= -104.47
  Rhamondre Stevenson    (RB)  predicted_KVS=  -21.88  keeper_cost=   49.86  raw_delta=  -71.74  need_z=+1.65  adjustment=  -35.74  ADJUSTED= -107.48

-- HarryUncle receives --
  Christian McCaffrey    (RB)  predicted_KVS=   74.67  keeper_cost=  107.36  raw_delta=  -32.70  need_z=+1.03  adjustment=  -22.42  ADJUSTED=  -55.11
      CAVEAT: low_confidence_extreme_delta: |vorp_delta_yoy| > 100 -- this region has confirmed, held-out degraded accuracy at every modeled position (see 08a-08d_shap_*.ipynb); treat the exact predicted_KVS number with real skepticism, not just the direction.
  Jaxson Dart       

**Real-data eyeball check, Trade 2**: both `low_confidence_extreme_delta` (McCaffrey) and `no_delta_history` (Dart) fire exactly as expected and are both present, correctly attributed to `team_b`/the right player, in the final `caveats` list -- neither one collapses into a flat, unattributed pile. Both totals land negative again (`team_a_total = -211.95`, `team_b_total = -54.24`), triggering the same `magnitude_relative_both_negative` fairness path -- this time on a real, dramatic gap (`20.4% / 79.6%`), correctly reading this as a lopsided trade heavily favoring HarryUncle rather than collapsing to a misleading 0%/100% the way the pre-fix shift math would have on values this large and this far apart. Top-mover selection worked correctly on the 2-player side too: both players on each side get explained here since `top_n_explained=2` is the full side, but the ranking by `|adjusted_net_kvs_delta|` (Stevenson's -107.48 edges out Jefferson's -104.47 on team A; McCaffrey's -55.11 dwarfs Dart's +0.87 on team B) is exactly the ordering `_select_top_movers` is supposed to produce. All four SHAP explanations ran with no `explanation_error`.

Cross-position confirmation, spotted while reading Dart's own explanation rather than by re-running a dedicated SHAP investigation: his `vorp_delta_yoy` is `NaN` (a true rookie, same as Tyler Warren in `08d_shap_te.ipynb`), and that missing-value branch contributes `+7.262` -- essentially the same sign and magnitude as Warren's `+8.1` in the TE model. Two different positions, two different trained models, and both show XGBoost's dedicated missing-value branch producing a real, non-trivial, similarly-sized positive contribution rather than treating a missing delta as zero -- the `no_delta_history` fix from earlier tonight is holding up end to end through the real trade engine, not just in the notebook it was originally found in. Worth noting this is also the first time this specific missing-value-branch behavior has been checked for QB at all -- `08a_shap_qb.ipynb` only ever investigated the *extreme-measured*-delta tail, not the no-history case -- so this is a new corroborating data point, not a rerun of an existing one.

## Summary

**`evaluate_trade` is real, not just wired-together in principle**: two actual trades between ten real rostered players ran end to end -- real `evaluate_player_trade_value` calls (real XGBoost models), real `apply_positional_need_adjustment` against real current rosters, real `compute_fairness_score`, and real `explain_player` SHAP breakdowns for the actual top movers on each side. Nothing in either run was mocked or injected; the dependency-injection seams (`evaluate_player_fn`/`explain_player_fn`) exist for `tests/test_evaluate_trade.py`'s orchestration tests, not for this notebook.

**A real gap got fixed along the way, not papered over**: Phase 5's `explain_player` existed only as four duplicated notebook cells before tonight; `src/explain/explain_player.py` is now the real, single, reusable, importable version `evaluate_trade.py` actually depends on -- and it reuses `net_value.py`'s own panel builders rather than re-deriving the same features a second, possibly-diverging way.

**Both fairness verdicts landed in the both-negative fallback path on real data**, not a contrived test case -- confirming that fallback (`magnitude_relative_both_negative`, from a bug found and fixed by directly checking a suspected `-100 vs -99` edge case) is not just a unit-test curiosity; it is the formula that actually fired for two real trades tonight, and it produced sensible, non-extreme percentage splits both times.

**Every caveat survived the full pipeline, correctly attributed**: `low_confidence_extreme_delta` (Terry McLaurin, Christian McCaffrey) and `no_delta_history` (Jaxson Dart) all made it from `net_value.py`'s per-player computation through `positional_need.py`'s adjustment and into `evaluate_trade`'s final, side-and-player-tagged `caveats` list -- exactly the "never silently dropped, never anonymized" standard every earlier module in this phase already held itself to.